# Load libraries

In [ ]:
import os
import gc

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import xgboost as xgb
from xgboost import XGBRegressor

from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

# Custom functions

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
def rmspe(y_true, y_pred):
    return  (np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))

In [ ]:
def feval_rmspe(y_pred, xgb_dtrain):
    y_true = xgb_dtrain.get_label()
    return "RMSPE", rmspe(y_true, y_pred)

# Read in data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

opt_train_df = pd.read_csv("/kaggle/input/optiver-training-data/optiver_train2.csv")

In [ ]:
display(train_df.head(2))
display(test_df.head(2))
display(submit_df.head(2))

In [ ]:
opt_train_df.shape

In [ ]:
opt_train_df.head()

In [ ]:
# opt_train_df['wap_range'] = opt_train_df['wap_min'] - opt_train_df['wap_max']
# opt_train_df['wap2_range'] = opt_train_df['wap2_min'] - opt_train_df['wap2_max']
# opt_train_df['bid_ask_diff_range'] = opt_train_df['bid_ask_diff_max'] - opt_train_df['bid_ask_diff_min']

# Data split

In [ ]:
opt_train_df['fold'] = -1

group_kfold = GroupKFold(n_splits=5)

kfold_val_dict = dict()

for fold, (_,val_idx) in enumerate(group_kfold.split(opt_train_df,opt_train_df['target'].values,opt_train_df['time_id'])):
    opt_train_df.loc[val_idx, 'fold'] = fold
    
    kfold_val_dict[fold] = val_idx

opt_train_df['fold'].value_counts()

# KFold training

In [ ]:
# Model parameters
xgb_params = {
 'tree_method': 'gpu_hist',
 'colsample_bytree': 0.5,
 'gamma': 0.12,
 'learning_rate': 0.01,
 'max_depth': 5,
#  'min_child_weight': 75,
 'reg_alpha': 0.05,
 'reg_lambda': 10,
 'n_estimators': 10000,
 'random_state': 101,
#  'scale_pos_weight': 6,
 'subsample': 0.85,
 'n_jobs': -1,
 'use_label_encoder': False
}

xgb_model = XGBRegressor(**xgb_params)

In [ ]:
opt_train_df.columns

In [ ]:
val_idx

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

for fold in tqdm(range(5)):

    print(f"Training fold {fold}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
    
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
     
    xgb_model = XGBRegressor(**xgb_params)
    xgb_model.fit(trn_x, trn_y,
            eval_set= [(val_x, val_y),(trn_x, trn_y)], 
            eval_metric=['rmse','mae'], 
#             eval_metric=feval_rmspe,
#             sample_weight = train_weights,
#             sample_weight_eval_set = val_weights,
            verbose=50, 
            early_stopping_rounds=150
           )    
    
    best_iteration = xgb_model.get_booster().best_ntree_limit
    
    print(f"Best iteration: {best_iteration}")
    
    var_imp[f'Fold{fold+1}'] = pd.Series(xgb_model.feature_importances_, index=trn_x.columns)
    
    oof_preds[val_idx] = xgb_model.predict(val_x, ntree_limit=best_iteration)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del xgb_model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = opt_train_df['real_vol1'].values),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = opt_train_df['real_vol1'].values),3)
print(f'R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
var_imp

In [ ]:
modelCols = var_imp[var_imp['Fold1']>0].index.tolist()
modelCols

# Try LGBM

In [ ]:
import lightgbm as lgb

In [ ]:
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
opt_train_df[modelCols].dtypes

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

for fold in tqdm(range(5)):

    print(f"Training fold {fold}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold].drop(columns=['time_id','target','fold'])[modelCols]
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold].drop(columns=['time_id','target','fold'])[modelCols]
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    # Root mean squared percentage error weights
    train_weights = 1 / np.square(trn_y)
    val_weights = 1 / np.square(val_y)
    train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)
    val_dataset = lgb.Dataset(val_x, val_y, weight = val_weights)
    model = lgb.train(params = params,
                      num_boost_round=10000,
                      train_set = train_dataset, 
                      valid_sets = [train_dataset, val_dataset], 
                      verbose_eval = 50,
                      early_stopping_rounds=50,
                      feval = feval_rmspe)
    
    
    var_imp[f'Fold{fold+1}'] = pd.Series(model.feature_importance(), index=trn_x.columns)
    
    oof_preds[val_idx] = model.predict(val_x)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
var_imp

In [ ]:
opt_train_df[modelCols].columns.tolist()

# Drop low importance variables

In [ ]:
# StratifiedGroupKFold - Run training
oof_preds = np.zeros(opt_train_df.shape[0])

var_imp = pd.DataFrame()

for fold in tqdm(range(5)):

    print(f"Training fold {fold}")
    
    trn_x = opt_train_df[opt_train_df['fold']!=fold][modelCols]
    trn_y = opt_train_df[opt_train_df['fold']!=fold]['target'].values
    
    val_x =opt_train_df[opt_train_df['fold']==fold][modelCols]
    val_y = opt_train_df[opt_train_df['fold']==fold]['target'].values
    
    val_idx = kfold_val_dict[fold]
     
    xgb_model = XGBRegressor(**xgb_params)
    xgb_model.fit(trn_x, trn_y,
            eval_set= [(trn_x, trn_y), (val_x, val_y)],
#             eval_metric=['rmse','mae'], 
            eval_metric=feval_rmspe,
            verbose=50, 
            early_stopping_rounds=150,
           )    
    
    best_iteration = xgb_model.get_booster().best_ntree_limit
    
    print(f"Best iteration: {best_iteration}")
    
    var_imp[f'Fold{fold+1}'] = pd.Series(xgb_model.feature_importances_, index=trn_x.columns)
    
    oof_preds[val_idx] = xgb_model.predict(val_x, ntree_limit=best_iteration)
    
    R2 = round(r2_score(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    RMSPE = round(rmspe(y_true = val_y, y_pred = oof_preds[val_idx]),3)
    print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')
    
    del xgb_model, trn_x, trn_y, val_x, val_y
    _ = gc.collect()

R2 = round(r2_score(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'].values, y_pred = oof_preds),3)
print(f'OOF: R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
dropCols = ['target']
X_train_temp = opt_train_df.drop(columns = dropCols)

y_train_temp = opt_train_df['target'].values.ravel()

print(f"Training data shape: {X_train_temp.shape}")

# Train model
xgb_model.fit(X_train_temp, y_train_temp, eval_metric=['rmse'], verbose=50) 
#               eval_set= [(X_train_temp, y_train_temp), (X_val_temp, y_val_temp)],
#             eval_metric=['auc','map'], verbose=250, 
#             early_stopping_rounds=150)

# best_iteration = xgb_model.get_booster().best_ntree_limit

# print(f"Best iteration: {best_iteration}")

In [ ]:
# Train performance
# ytrain_pred_prob_xgb = xgb_model.predict_proba(X_train_temp, ntree_limit=best_iteration)
ytrain_pred_xgb = xgb_model.predict(X_train_temp)
R2 = round(r2_score(y_true = y_train_temp, y_pred = ytrain_pred_xgb),3)
RMSPE = round(rmspe(y_true = y_train_temp, y_pred = ytrain_pred_xgb),3)
print(f'Performance of the naive prediction: R2 score: {R2}, RMSPE: {RMSPE}')

# Final training

In [ ]:
modelCols = ['wapbal_sum',
             'real_vol1',
             'real_vol2',
             'real_vol3',
             'real_vol4',
             'spread_sum',
             'bid_ask_diff_sum']

In [ ]:
seed0=2021
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': 115,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
#     'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'device':'gpu',
    'verbose': -1}

In [ ]:
trn_x = opt_train_df[modelCols]
trn_y = opt_train_df['target'].values

# Root mean squared percentage error weights
train_weights = 1 / np.square(trn_y)
train_dataset = lgb.Dataset(trn_x, trn_y, weight = train_weights)

model = lgb.train(params = params,
#                   num_boost_round=10000,
                  train_set = train_dataset, 
                  valid_sets = [train_dataset], 
                  verbose_eval = 50,
                  feval = feval_rmspe)

In [ ]:
pd.Series(model.feature_importance(), index=trn_x.columns)

In [ ]:
preds = model.predict(trn_x)

R2 = round(r2_score(y_true = trn_y, y_pred = preds),3)
RMSPE = round(rmspe(y_true = trn_y, y_pred = preds),3)
print(f'Fold {fold+1}: R2 score: {R2}, RMSPE: {RMSPE}')

In [ ]:
pickle_dump("./lgbm_trail1.pkl", model)